In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit


In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)


def stratified_sample(df, obj_var, n_samples, q, seed):
    if len(df) > n_samples + q:
        df = df.copy()
        df["bin"] = pd.qcut(df[obj_var], q=q, labels=False, duplicates="drop")

        sss = StratifiedShuffleSplit(
            n_splits=1, train_size=n_samples, random_state=seed
        )
        train_idx, _ = next(sss.split(df.index.to_frame(), df["bin"]))
        df_extracted = df.iloc[train_idx].drop(columns="bin")
    else:
        df_extracted = df.copy()

    return df_extracted


def generate_pair(df, obj_var, repeat):
    id_arr = df["ID"].to_numpy()
    id_arr = np.tile(id_arr, repeat)
    id_arr = np.random.permutation(id_arr)

    id_pair_arr = np.stack([id_arr, np.random.permutation(id_arr)], axis=1)

    # Delete same ID pairs
    id_pair_arr = np.array([x for x in id_pair_arr if x[0] != x[1]])

    # Delete duplicate pairs
    seen = []
    id_pair_arr = np.array(
        [x for x in id_pair_arr if set(x) not in seen and not seen.append(set(x))]
    )

    # Delete pairs with same objective variable value
    id_pair_arr = np.array(
        [
            x
            for x in id_pair_arr
            if df.loc[df["ID"] == x[0], obj_var].values[0]
            != df.loc[df["ID"] == x[1], obj_var].values[0]
        ]
    )

    # Make pairs balanced for objective variable comparison
    id_pair_arr_a_lt_b = np.array(
        [
            x
            for x in id_pair_arr
            if df.loc[df["ID"] == x[0], obj_var].values[0]
            < df.loc[df["ID"] == x[1], obj_var].values[0]
        ]
    )
    id_pair_arr_a_gt_b = np.array(
        [
            x
            for x in id_pair_arr
            if df.loc[df["ID"] == x[0], obj_var].values[0]
            > df.loc[df["ID"] == x[1], obj_var].values[0]
        ]
    )
    n_min = min(len(id_pair_arr_a_lt_b), len(id_pair_arr_a_gt_b))
    id_pair_arr = np.vstack([id_pair_arr_a_lt_b[:n_min], id_pair_arr_a_gt_b[:n_min]])

    # Make DataFrame
    df_pair = pd.DataFrame(id_pair_arr, columns=["ID_A", "ID_B"])
    df_a = df.copy()
    df_a.columns = [f"{c}_A" for c in df_a.columns.to_list()]
    df_b = df.copy()
    df_b.columns = [f"{c}_B" for c in df_b.columns.to_list()]

    df_pair = pd.merge(df_pair, df_a, how="left", on="ID_A")
    df_pair = pd.merge(df_pair, df_b, how="left", on="ID_B")

    df_pair_wo_obj = df_pair.drop(columns=[f"{obj_var}_A", f"{obj_var}_B"])

    return df_pair, df_pair_wo_obj


def process_dataset(
    dataset_name,
    obj_var,
    n_samples=150,
    q=10,
    repeat=4,
    dataset_dir_path=Path("./dataset"),
    dataset_save_path=Path("./dataset_processed"),
    seed=42,
):
    set_seed(seed)

    # Read dataset
    df = pd.read_csv(dataset_dir_path / f"dataset_{dataset_name}.csv").astype(
        {"ID": int}
    )
    print(df.info())
    display(df.head(5))
    display(df[obj_var].describe())

    # Randomly extract data so that the distribution of the objective variable is the same
    df_extracted = (
        stratified_sample(df=df, obj_var=obj_var, n_samples=n_samples, q=q, seed=seed)
        .sort_values("ID")
        .reset_index(drop=True)
    )

    # Check changes
    print(f"num of samples: {len(df)} -> {len(df_extracted)}")
    print("Unique counts for each column:")
    for col in df.columns.tolist():
        print(f"{col}: {df[col].nunique()} -> {df_extracted[col].nunique()}")

    # Visualize distribution
    plt.figure(figsize=(12, 4))

    # Histogram
    plt.subplot(1, 2, 1)
    sns.histplot(df[obj_var], bins=n_samples // 2, kde=False, label="Original Data")
    sns.histplot(
        df_extracted[obj_var], bins=n_samples // 2, kde=False, label="Extracted Data"
    )
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.xlabel("Objective Variable", fontsize=10)
    plt.ylabel("Frequency", fontsize=10)
    plt.legend(fontsize=10)

    # KDE
    plt.subplot(1, 2, 2)
    sns.kdeplot(df[obj_var], label="Original Data", fill=True)
    sns.kdeplot(df_extracted[obj_var], label="Extracted Data", fill=True)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.xlabel("Objective Variable", fontsize=10)
    plt.ylabel("Density", fontsize=10)
    plt.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

    # Save extracted data
    df_extracted.to_csv(
        dataset_save_path / f"dataset_{dataset_name}_extracted.csv", index=False
    )

    # Generate pairs
    df_pair, df_pair_wo_obj = generate_pair(
        df=df_extracted, obj_var=obj_var, repeat=repeat
    )
    print(f"num of pairs: {len(df_pair)}")

    # Save pairs
    df_pair.to_csv(dataset_save_path / f"dataset_{dataset_name}_pair.csv", index=False)
    df_pair_wo_obj.to_csv(
        dataset_save_path / f"dataset_{dataset_name}_pair_wo_obj.csv", index=False
    )

## Shields et al. (2021)

In [ ]:
process_dataset(dataset_name="BH_1", obj_var="yield")

In [ ]:
process_dataset(dataset_name="DA", obj_var="yield")

## Olympus

In [ ]:
process_dataset(dataset_name="alkox", obj_var="conversion")

In [ ]:
process_dataset(dataset_name="oer_plate_a", obj_var="overpotential")

In [ ]:
process_dataset(dataset_name="p3ht", obj_var="conductivity")

In [ ]:
process_dataset(dataset_name="photo_pce10", obj_var="degradation")

In [ ]:
process_dataset(dataset_name="photo_wf3", obj_var="degradation")

In [ ]:
process_dataset(dataset_name="suzuki", obj_var="yield")

In [ ]:
process_dataset(dataset_name="suzuki_edbo", obj_var="yield")